# Supervised Research (Law, Economics, and Data Science)
#### Project name "Anomaly Detection for CNP Transactions".

Student: Anna Kravchenko

akravchenko@student.ethz.ch

### Part 3 - Data preprocessing

## Goal

This notebook performs the preprocessing stage for fraud detection modeling on the labeled training data only.


The preprocessing focuses on:

- loading and merging `transaction` + `identity` tables (labeled data only),
- building a consistent `event_time` from `TransactionDT`,
- computing the chronological train/val/holdout split first, before any statistic is fit, everything downstream (rare-category vocab, frequency encodings, UID aggregates, V-block PCA) is fit on the `train` window only and applied (transform-only) to `val`/`holdout`,
- consuming the feature audit from Part 2 (the 339 raw `V*` columns are replaced by `Vgrp*_pca1` reductions, fit fresh here on the train split only)
- building two parallel encoding branches from the same engineered features: a label-encoded branch for gradient-boosted trees (order-invariant, fine for GBMs), and a one-hot + frequency-encoded + standardized branch for the distance/reconstruction-based models (Isolation Forest, Autoencoder), where raw integer label codes would impose a meaningless ordinal structure.

*Reproducibility notes.*
All fitted statistics (rare-category keep-lists, frequency counts, UID aggregates, V-block PCA components, one-hot vocabularies, scaler mean/std) are fit on split = train rows only and then applied to `val`/`holdout` without refitting. Unseen categories in `val`/`holdout` fall back to an explicit `__UNSEEN__`/`__OTHER__` bucket rather than being silently dropped or crashing the pipeline.

### Imports and config

In [3]:
import re
import gc
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option('display.max_rows', 120)
pd.set_option('display.max_columns', 300)

### Paths

In [4]:
#local folder
PROJECT_DIR = Path("/Users/kravchan/Downloads/Supervised Research/project")
DATA_DIR = PROJECT_DIR / 'Data'
ART_DIR = PROJECT_DIR / 'artifacts'
OUT_DIR = PROJECT_DIR / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('DATA_DIR exists:', DATA_DIR.exists())
print('ART_DIR exists:', ART_DIR.exists())

PROJECT_DIR: /Users/kravchan/Downloads/Supervised Research/project
DATA_DIR exists: True
ART_DIR exists: True


### Load labeled data only

Only `train_transaction.csv` and `train_identity.csv` are read.

The raw `train_transaction.csv` is ~683MB, so it is read in chunks with dtypes fixed up-front (numeric columns as `float32`/`int32`, text columns as `category`) to keep peak memory bounded, but this has no effect on the resulting data, it is just a memory efficient way to read it.

In [5]:
def fast_read_csv(path, id_cols=('TransactionID', 'isFraud'), chunksize=100_000):
    """Memory-efficient CSV reader: infers a compact dtype map from a header sample,
    then reads in chunks, casting text columns to `category` as it goes."""
    #a small sample first just to decide compact dtypes
    head = pd.read_csv(path, nrows=20_000)
    dtypes = {}
    for c in head.columns: 
        if c in id_cols:
            dtypes[c] = 'int32' #TransactionID and isFraud (id-like) get int32 type instead of the default int64
        elif head[c].dtype.kind in 'fi':
            dtypes[c] = 'float32' #numeric columns get float32 instead of pandas's default float64 
        else:
            dtypes[c] = 'object' #text columns are left as generic object type here (they get converted to the more efficient category type below)
    del head #the 20k row sample is no longer needed once the dtype map is built, so it is deleted

    #reads the file in chunks of 100k rows instead of all at once
    chunks = []
    for chunk in pd.read_csv(path, dtype=dtypes, chunksize=chunksize):
        for c, dt in dtypes.items():
            if dt == 'object':
                #converts text columns to pandas's category type as each chunk is read in (more memory-efficient than plain strings)
                chunk[c] = chunk[c].astype('category')
        chunks.append(chunk)
    df = pd.concat(chunks, ignore_index=True)  #glues all the chunks back together into one full df 
    del chunks #frees the list of individual chunks after combining into df
    gc.collect()
    return df


#times how long it takes to load and merge both files
t0 = time.time()
train_tr = fast_read_csv(DATA_DIR / 'train_transaction.csv')
train_id = fast_read_csv(DATA_DIR / 'train_identity.csv')
train = train_tr.merge(train_id, on='TransactionID', how='left')
del train_tr, train_id
gc.collect()

print('train (labeled only):', train.shape, f'({time.time()-t0:.1f}s)')

train (labeled only): (590540, 434) (6.0s)


load+merge completing in 6 seconds.

### `event_time` and calendar features

`TransactionDT` is a relative time variable (seconds from an unknown reference point). It is converted into an approximate timestamp `event_time`, from which hour/day-of-week/day/month/week-of-year features are derived.

In [6]:
def ensure_event_time(df, epoch='2017-12-01'):
    if 'event_time' in df.columns:
        return df
    #column can be named differently after a merge, like TransactionDT_x
    tdt_candidates = [c for c in df.columns if c.startswith('TransactionDT')]
    if not tdt_candidates:
        raise KeyError('No TransactionDT-like column found.')
    tdt = tdt_candidates[0]
    df[tdt] = pd.to_numeric(df[tdt], errors='coerce')
    #TransactionDT is seconds since a hidden reference date, so this turns it into a real timestamp we can use
    df['event_time'] = pd.Timestamp(epoch) + pd.to_timedelta(df[tdt], unit='s')
    return df

train = ensure_event_time(train)
#pulls hour/day/month etc out of the timestamp as separate columns the model can use
dt = train['event_time']
train['event_hour'] = dt.dt.hour.astype('int16')
train['event_dow'] = dt.dt.dayofweek.astype('int16')
train['event_day'] = dt.dt.day.astype('int16')
train['event_month'] = dt.dt.month.astype('int16')
train['event_weekofyear'] = dt.dt.isocalendar().week.astype('int16')

print(train[['event_time','event_hour','event_dow','event_month']].head())

           event_time  event_hour  event_dow  event_month
0 2017-12-02 00:00:00           0          5           12
1 2017-12-02 00:00:01           0          5           12
2 2017-12-02 00:01:09           0          5           12
3 2017-12-02 00:01:39           0          5           12
4 2017-12-02 00:01:46           0          5           12


### Chronological split (computed before any statistic is fit)

The split is the first derived feature here, and every encoding step below explicitly fits on `split == "train"` rows only, nothing downstream sees `val`/`holdout` rows until the transform step.

The split boundaries (`q60`/`q80` date quantiles) are reused from the EDA/feature-audit artifact (`split_config.json`) so all four notebooks share identical train/val/holdout windows. This third window is the test - `holdout`, its labels are never touched by any fitting or selection decision.

In [7]:
#reuse the same cutoffs saved in part 1/2
split_fp = ART_DIR / 'split_config.json'
if split_fp.exists():
    split_cfg = json.loads(split_fp.read_text())
    q60 = pd.to_datetime(split_cfg['q60'])
    q80 = pd.to_datetime(split_cfg['q80'])
else:
    q60 = train['event_time'].quantile(0.60)
    q80 = train['event_time'].quantile(0.80)

#everything before q60 is train, between q60 and q80 is val, after q80 is holdout
train['split'] = np.where(train['event_time'] < q60, 'train',
                  np.where(train['event_time'] < q80, 'val', 'holdout'))

#True/False mask for train rows, used everywhere below so only train rows are fit on
is_train = (train['split'] == 'train').values

print(train['split'].value_counts())
print('q60:', q60, 'q80:', q80)
print('fit rows (train split only):', int(is_train.sum()))

split
train      354324
val        118108
holdout    118108
Name: count, dtype: int64
q60: 2018-03-12 05:23:02.400000 q80: 2018-04-21 02:54:13.600000
fit rows (train split only): 354324


Everything matches previous parts; `is_train` is the boolean mask used everywhere below to make sure only training-window rows contribute to any fitted statistic.

### Email and device normalization

String parsing (provider/TLD extraction, device brand/platform regex). No statistics are fit here, so there's nothing specifically about splitting in this step.

In [8]:
PROV_MAP = { #maps raw email domains to a shorter provider name
    'gmail.com':'gmail', 'googlemail.com':'gmail', 'yahoo.com':'yahoo', 'yahoo.co.uk':'yahoo',
    'hotmail.com':'hotmail', 'outlook.com':'outlook', 'outlook.es':'outlook', 'live.com':'live',
    'icloud.com':'icloud', 'aol.com':'aol', 'msn.com':'msn', 'protonmail.com':'proton',
    'yandex.ru':'yandex', 'yandex.com':'yandex', 'mail.com':'mail', 'gmx.com':'gmx'
}
FREE_SET = {'gmail','yahoo','hotmail','outlook','live','icloud','aol','proton','yandex','mail','gmx','msn'}

def add_email_features(df):
    for side in ['P_emaildomain', 'R_emaildomain']:
        if side in df.columns:
            dom = df[side].astype(str).str.lower()
            #looks up the short provider name, if it is not in the map, falls back to the first part of the domain
            prov = dom.map(PROV_MAP).fillna(dom.str.split('.').str[0].replace('nan', 'unknown'))
            tld = dom.str.split('.').str[-1].replace('nan', 'unknown')
            df[side + '_prov'] = prov.astype(str)
            df[side + '_tld'] = tld.astype(str)
            df[side + '_is_free'] = prov.isin(FREE_SET).astype('int8') #flags whether this is a known free email provider
    return df

def _norm_device(s): #lowercases and cleans up whitespace so the same device string is not counted twice due to formatting
    s = str(s).lower()
    return re.sub(r'\s+', ' ', s).strip()

def add_device_features(df):
    if 'DeviceInfo' in df.columns:
        di = df['DeviceInfo'].astype(str).fillna('unknown').map(_norm_device)
        df['device_brand'] = di.str.extract( #pulls out a known name from the device string if there is one, otherwise labels it "other"
            r'(samsung|huawei|motorola|sony|lg|asus|xiaomi|oneplus|lenovo|nexus|iphone|ipad|mac|windows|linux)',
            expand=False).fillna('other').astype(str)
        #groups devices into apple/windows/linux/android based on keywords in the device string
        df['device_platform'] = np.where(di.str.contains('iphone|ipad|mac|ios'), 'apple',
                                 np.where(di.str.contains('windows'), 'windows',
                                 np.where(di.str.contains('linux'), 'linux', 'android/other')))
        df['device_platform'] = df['device_platform'].astype(str)
    return df

train = add_email_features(train)
train = add_device_features(train)
print('email/device features added:', [c for c in train.columns if '_prov' in c or '_tld' in c or '_is_free' in c or 'device_' in c])

email/device features added: ['P_emaildomain_prov', 'P_emaildomain_tld', 'P_emaildomain_is_free', 'R_emaildomain_prov', 'R_emaildomain_tld', 'R_emaildomain_is_free', 'device_brand', 'device_platform']


### UID / entity features

Combinations of card/address/email attributes, approximating a persistent customer/device entity. No fitting.

In [9]:
for c in ['card1','card2','card3','card5','addr1','addr2','P_emaildomain']:
    if c in train.columns:
        train[c] = train[c].astype(str) #casts these to strings first so they can be concatenated together into a single uid string below

#builds a rough same user id by combining card + address fields, since there is no real user id in this data
train['uid1'] = train['card1'] + '_' + train['card2'] + '_' + train['addr1']
train['uid2'] = train['card1'] + '_' + train['addr1'] + '_' + train['P_emaildomain']
train['uid3'] = train['card1'] + '_' + train['card2'] + '_' + train['addr1'] + '_' + train['addr2']

for u in ['uid1','uid2','uid3']:
    print(u, train[u].nunique(), 'unique values')

uid1 41672 unique values
uid2 90375 unique values
uid3 41724 unique values


### Rare-category bucketing (fit on `train` split only)

Frequent categories (>=0.1% of training-window rows) are kept; everything else collapses into an `__OTHER__` bucket, and missing values get their own `__MISS__` label. Fitting the keep-list on `train` only means the category vocabulary itself cannot be influenced by rows that are chronologically in the future relative to the training window.

In [10]:
#finds which category values are common enough (fit on train split only) to keep as-is, everything else gets grouped into __OTHER__ later
def fit_rare_keep(df, cols, is_train_mask, min_freq=0.001):
    rare_keep = {}
    for c in cols:
        if c not in df.columns:
            continue
        s = df.loc[is_train_mask, c]
        s = s.astype(object).where(s.notna(), '__MISS__').astype(str) #missings get their own label so they count as a category too
        freq = s.value_counts(normalize=True) #share of rows each category value makes up, on train only
        keep = freq[freq >= min_freq].index.astype(str).tolist()
        rare_keep[c] = {'min_freq': min_freq, 'keep_values': keep,
                         'fill_label': '__OTHER__', 'missing_label': '__MISS__'}
    return rare_keep

#applies the train-fit list to any split, so val/holdout never leak into the definition of "common"
def apply_rare_bucketing(df, rare_cfg):
    for col, cfg in rare_cfg.items():
        if col not in df.columns:
            continue
        s = df[col].astype(object).where(df[col].notna(), cfg['missing_label']).astype(str)
        keep = set(cfg['keep_values'])
        #keeps the value if it was common in train, otherwise relabels it as OTHER
        df[col + '_grp'] = np.where(s.isin(keep), s, cfg['fill_label'])
    return df

rare_cols = ['ProductCD', 'card4', 'card6', 'P_emaildomain_prov', 'device_brand', 'device_platform', 'DeviceType']
rare_keep = fit_rare_keep(train, rare_cols, is_train)
train = apply_rare_bucketing(train, rare_keep)
(ART_DIR / 'rare_bucket_keep_p3.json').write_text(json.dumps(rare_keep, indent=2))
print('rare-bucketing fit on train split, applied to', len(rare_keep), 'columns')

rare-bucketing fit on train split, applied to 7 columns


### Frequency encoding (fit on `train` split only)

Frequency of an identifier can be predictive (very common vs. rare vs. one-off entities behave differently). Counts are computed from `train` window rows only; `val`/`holdout` values not seen in the training window map to a count of 0 rather than pulling in future window frequency information.

In [11]:
freq_cols = [
    'card1','card2','card3','card5','addr1','addr2',
    'P_emaildomain','R_emaildomain','uid1','uid2','uid3',
    'device_brand','device_platform','DeviceType',
    'id_30','id_31','id_33','DeviceInfo','P_emaildomain_prov',
]
freq_cols = [c for c in freq_cols if c in train.columns]

freq_maps = {}
#counts how often each category value shows up in the train split only, then uses that count itself as a new numeric feature
for c in freq_cols:
    vc = train.loc[is_train, c].astype(str).value_counts()
    freq_maps[c] = vc
    train[c + '_fe'] = train[c].astype(str).map(vc).fillna(0).astype('float32')

print('frequency-encoded (train-fit):', len(freq_cols), 'columns')

frequency-encoded (train-fit): 19 columns


### UID aggregate features (fit on `train` split only)

Mean/std of `TransactionAmt` per UID, can help by telling the model whether an amount is typical or unusual for that entity. The aggregate is fit exclusively on `train` window rows, so a UID's `val`/`holdout` rows can never leak their own future amounts backward into earlier rows; UIDs unseen in the training window fall back to the training-window global mean/std of `TransactionAmt`.

In [13]:
#overall train average/spread of TransactionAmt, used as a fallback
global_amt_mean = train.loc[is_train, 'TransactionAmt'].mean()
global_amt_std = train.loc[is_train, 'TransactionAmt'].std()

uid_agg = {}
for uid in ['uid1', 'uid2', 'uid3']:
    #average and spread of TransactionAmt per uid, computed on train rows only
    g = train.loc[is_train].groupby(uid)['TransactionAmt'].agg(['mean', 'std'])
    uid_agg[uid] = g
    #maps each row's uid to the train only mean/std computed for that uid
    m = train[uid].map(g['mean']).astype('float32')
    s = train[uid].map(g['std']).astype('float32')
    #if a uid only shows up in val/holdout and was never seen in train, falls back to the overall train average instead of leaving it empty
    train[f'{uid}_TransactionAmt_mean'] = m.fillna(global_amt_mean).astype('float32')
    train[f'{uid}_TransactionAmt_std'] = s.fillna(global_amt_std).fillna(0).astype('float32')

print('UID aggregate features added for', len(uid_agg), 'UIDs (fit on train split only)')

UID aggregate features added for 3 UIDs (fit on train split only)


### V-block reduction (consuming Part 2's audit)

Part 2's NaN-pattern grouping (`v_groups.json`, 11 groups covering the 339 `V*` columns) is reused here, but the reduction itself is refit from scratch: one principal component per group, with the PCA **fit on `train`-split rows only** (median-imputed using the train-split median). A single fixed method (PCA1), fit without touching any label, keeps this leakage-safe and reproducible across groups. The raw 339 `V*` columns are then dropped and replaced by the 11 `Vgrp*_pca1` columns, so the V-block audit work in Part 2 is actually consumed here rather than left unused.

In [14]:
from sklearn.decomposition import PCA

v_groups = json.loads((ART_DIR / 'v_groups.json').read_text()) #loads the V column groupings from part 2
vgrp_cols = []
for gi, g in enumerate(v_groups, 1):
    sub = [c for c in g if c in train.columns]
    if len(sub) < 2:
        continue
    med = train.loc[is_train, sub].median() #train only median used to fill gaps before PCA
    X = train[sub].fillna(med).astype('float32').values  #PCA cannot handle NaNs, so missing values are filled in first
    pca = PCA(n_components=1, random_state=0)
    pca.fit(X[is_train]) #PCA is fit on train rows only, then used to transform every row, so val/holdout never influence the components
    #compresses this whole V-group into one column
    train[f'Vgrp{gi:03d}_pca1'] = pca.transform(X)[:, 0].astype('float32')
    vgrp_cols.append(f'Vgrp{gi:03d}_pca1')

raw_v_cols = [c for c in train.columns if c.startswith('V') and not c.startswith('Vgrp') and c[1:].isdigit()]
#raw V columns can be dropped now that each group is replaced by its single PCA component
train = train.drop(columns=raw_v_cols)

print(f'V-block reduced: dropped {len(raw_v_cols)} raw V columns, added {len(vgrp_cols)} Vgrp*_pca1 columns')

V-block reduced: dropped 339 raw V columns, added 11 Vgrp*_pca1 columns


(matching Part 2's 11 NaN-pattern groups exactly).

### Missing-value indicators

Part 1/2 established that missingness is often informative (not random) for several `id_*`/`D*`/`dist*` columns. Explicit `_isna` flags are added for columns whose missing share (measured on the `train` split) falls in an "informative" band, neither near-universal-missing nor near-never-missing, where a flag adds little signal either way.

In [15]:
miss_share = train.loc[is_train].isna().mean() #share of missing values per column on train
#keeps columns missing 5%-95% of the time
informative_missing = miss_share[(miss_share > 0.05) & (miss_share < 0.95)].index.tolist()
informative_missing = [c for c in informative_missing if c != 'split']

#builds all the missings indicator columns at once in a separate df
isna_df = pd.DataFrame(
    {c + '_isna': train[c].isna().astype('int8') for c in informative_missing},
    index=train.index,
)
train = pd.concat([train, isna_df], axis=1) #joins them in one go
#adding one column at a time here was much slower

print(f'added {len(informative_missing)} missing-value indicator columns (5%-95% missing on train split)')

added 57 missing-value indicator columns (5%-95% missing on train split)


### Drop ultra-sparse columns

Columns with >98% missingness *on the train split* (protecting `TransactionID`/`isFraud`/`event_time`/`split`) are dropped -- unlikely to be reliable, and now redundant with their `_isna` flag where one was informative enough to keep.

In [16]:
keep_special = {'TransactionID', 'isFraud', 'event_time', 'split'}
#drops columns missing more than 98% of the time on train, there is almost nothing left to learn from
to_drop = [c for c in train.columns if c not in keep_special and miss_share.get(c, 0) > 0.98]
train = train.drop(columns=to_drop)
print('dropped', len(to_drop), 'ultra-sparse (>98% missing on train) columns:', to_drop)
print('shape after V-block reduction + sparse-column drop:', train.shape)

dropped 9 ultra-sparse (>98% missing on train) columns: ['id_07', 'id_08', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27']
shape after V-block reduction + sparse-column drop: (590540, 204)


### Drop near-constant columns (consuming Part 2's audit)

Part 2 identifies near-constant columns (>99.5% single value: `C3` plus several raw `V*` columns) and exports them as `dropped_near_constant_cols` in `selected_features.json`. Most of those are raw `V*` columns already removed by the V-block reduction above; whatever's left (in practice just `C3`) is dropped here too.

In [17]:
sel_fp = ART_DIR / 'selected_features.json'
if sel_fp.exists():
    selected = json.loads(sel_fp.read_text())
    #near-constant columns flagged in part 2 audit that are still around here
    near_constant_remaining = [c for c in selected.get('dropped_near_constant_cols', []) if c in train.columns]
    if near_constant_remaining:
        train = train.drop(columns=near_constant_remaining)
    print('near-constant columns dropped (from Part 2 audit, after V-block reduction already removed the rest):',
          near_constant_remaining)
else:
    print('selected_features.json not found -- skipping (run Part 2 first)')

near-constant columns dropped (from Part 2 audit, after V-block reduction already removed the rest): ['C3']


## Two encoding branches

Raw integer label-encoding is fine for LightGBM/CatBoost -- tree splits don't care whether `card4=3` is "close to" `card4=2`. It is actively harmful for Isolation Forest and an Autoencoder, which are distance/reconstruction-based: an arbitrary integer code implies an ordinal/metric relationship between categories that isn't real. So from this point on, two parallel feature sets are built from the same engineered columns above:

- Branch A - GBM-ready (`train_processed_gbm.parquet`): categoricals label-encoded, numeric features left on their natural scale. Intended for a calibrated LightGBM supervised reference model.
- Branch B - distance-model-ready (`train_processed_ifae.parquet`): low-cardinality categoricals one-hot encoded, high-cardinality categoricals represented only via their (already leakage-safe) frequency encoding, and every numeric feature median-imputed + standardized. Intended for distance/reconstruction-based models such as Isolation Forest and an autoencoder.

Both branches fit everything (label vocabularies, one-hot categories, imputation medians, scaler mean/std) on split = train rows only.

### Branch A (GBM-ready (label encoding, train-fit vocab + `__UNSEEN__` bucket))

In [18]:
exclude = {'TransactionID', 'isFraud', 'event_time', 'split'} #these columns are never used as model features, just kept

#using the rare-bucketed (_grp) version instead of the raw column for the 7 columns that were bucketed
grp_pairs = {'ProductCD_grp':'ProductCD', 'card4_grp':'card4', 'card6_grp':'card6',
             'P_emaildomain_prov_grp':'P_emaildomain_prov', 'device_brand_grp':'device_brand',
             'device_platform_grp':'device_platform', 'DeviceType_grp':'DeviceType'}

#swaps in the rare-bucketed version of these 7 columns
#drops the raw version, so the model only ever sees the bucketed categories
gbm = train.drop(columns=list(grp_pairs.values())).rename(columns=grp_pairs)

#picks out the text/category columns (those need to become numbers for the GBM)
cat_cols_gbm = [c for c in gbm.columns if c not in exclude and
                (gbm[c].dtype == 'object' or str(gbm[c].dtype) == 'category')]

UNSEEN = '__UNSEEN__'
gbm_label_maps = {}
#builds the category to number mapping using train rows only, 
#plus one extra "unseen" code for any category that shows up in val/holdout but never in train
for c in cat_cols_gbm:
    vocab = sorted(gbm.loc[is_train, c].astype(str).unique().tolist()) + [UNSEEN]
    mapping = {v: i for i, v in enumerate(vocab)}
    gbm_label_maps[c] = mapping
    codes = gbm[c].astype(str).map(mapping).fillna(mapping[UNSEEN]).astype('int32') #turns each category into its number code
    gbm[c] = codes #anything not seen in train becomes the unseen code

print('Branch A (GBM) shape:', gbm.shape, '|', len(cat_cols_gbm), 'label-encoded categoricals')

Branch A (GBM) shape: (590540, 196) | 44 label-encoded categoricals


### Branch B (distance-model-ready (one-hot + frequency encoding + standardization))

In [19]:
ifae = train.copy()

#a few extra high-cardinality columns that were not in the frequency-encoding pass above
for c in ['id_30', 'id_31', 'id_33', 'DeviceInfo', 'P_emaildomain_prov']:
    if c in ifae.columns and c + '_fe' not in ifae.columns:
        vc = ifae.loc[is_train, c].astype(str).value_counts()
        ifae[c + '_fe'] = ifae[c].astype(str).map(vc).fillna(0).astype('float32')

#drop raw high-cardinality text columns represented only via their *_fe frequency encoding from here on
raw_highcard_drop = ['card1','card2','card3','card5','addr1','addr2','P_emaildomain','R_emaildomain',
                      'uid1','uid2','uid3','DeviceInfo','id_30','id_31','id_33',
                      'device_brand','device_platform','DeviceType','P_emaildomain_prov']
ifae = ifae.drop(columns=[c for c in raw_highcard_drop if c in ifae.columns])

#remaining categoricals are low-cardinality -> one-hot, fit categories on train split only
low_card_cols = [c for c in ifae.columns if c not in exclude and
                  (ifae[c].dtype == 'object' or str(ifae[c].dtype) == 'category')]

ohe_frames, ohe_cats = [], {}
for c in low_card_cols:
    cats = sorted(ifae.loc[is_train, c].astype(str).unique().tolist())
    ohe_cats[c] = cats
    s = ifae[c].astype(str)
    #one-hot encodes using only the categories seen in train, plus one extra unseen category column for anything new in val/holdout
    dummies = pd.get_dummies(pd.Categorical(s, categories=cats + ['__UNSEEN__']), prefix=c, dtype='int8')
    unseen_mask = ~s.isin(cats)
    if unseen_mask.any():
        dummies.loc[unseen_mask, :] = 0  # unseen category -> all-zero indicator row, not a crash
    ohe_frames.append(dummies.astype('int8'))

meta_cols = ifae[['TransactionID', 'isFraud', 'event_time', 'split']].copy()
num_cols = [c for c in ifae.columns if c not in exclude and c not in low_card_cols]
X = ifae[num_cols].astype('float32').values

##median-impute (train-fit) then standardize (train-fit mean/std)
#fills missing numeric values with the train-only median before scaling (models cannot handle NaNs directly)
med = np.nanmedian(X[is_train], axis=0)
#finds the row/column positions of every NaN in the numeric matrix
nan_idx = np.where(np.isnan(X))
if len(nan_idx[0]):
    #fills each NaN with the median for its own column, using the column indices from nan_idx to look up the right median value
    X[nan_idx] = np.take(med, nan_idx[1])
mean = X[is_train].mean(axis=0)
std = X[is_train].std(axis=0)
std[std == 0] = 1.0
#standardizes using train-only mean/std
X = ((X - mean) / std).astype('float32')
##so val/holdout are scaled the same way without leaking their own statistics in

num_df = pd.DataFrame(X, columns=num_cols, index=meta_cols.index)
ohe_block = pd.concat(ohe_frames, axis=1) if ohe_frames else pd.DataFrame(index=meta_cols.index)
#stitches the id/label columns, the scaled numeric columns, and the one-hot columns back together into a single df
ifae = pd.concat([meta_cols, num_df, ohe_block], axis=1)

print('Branch B (Isolation Forest / Autoencoder) shape:', ifae.shape,
      '|', len(low_card_cols), 'one-hot columns,', len(num_cols), 'scaled numeric columns')

Branch B (Isolation Forest / Autoencoder) shape: (590540, 377) | 32 one-hot columns, 148 scaled numeric columns


Branch A (GBM) came out to **(590540, 192)** columns, Branch B (IF/AE) to **(590540, 377)** columns after one-hot expansion -- both a large reduction from the original single 434-column table (mostly from replacing 339 raw `V*` columns with 11 `Vgrp*_pca1` columns), and both built without touching the Kaggle test file or leaking `val`/`holdout` rows into any fitted statistic.

### Save processed data and fit artifacts

In [20]:
train_out = train[['TransactionID','isFraud','event_time','split']].copy()
train_out['event_time'] = train_out['event_time'].astype(str)

gbm_out = gbm.copy() 
gbm_out['event_time'] = gbm_out['event_time'].astype(str) 
##converts the timestamp to plain text before saving to parquet, it reads back the same way regardless of environment
gbm_out.to_parquet(OUT_DIR / 'train_processed_gbm.parquet', index=False)

ifae_out = ifae.copy()
ifae_out['event_time'] = ifae_out['event_time'].astype(str)
ifae_out.to_parquet(OUT_DIR / 'train_processed_ifae.parquet', index=False)

import joblib
#saves every fitted encoder/statistic used above (rare-category lists, frequency maps, UID aggregates, label maps, one-hot categories, scaler/imputer values) so the exact same transformations could be reapplied later if needed
joblib.dump({'rare_keep': rare_keep, 'freq_maps': {k: v.to_dict() for k, v in freq_maps.items()},
             'uid_agg': uid_agg, 'gbm_label_maps': gbm_label_maps, 'ohe_cats': ohe_cats,
             'scaler_mean': mean.tolist(), 'scaler_std': std.tolist(), 'impute_median': med.tolist(),
             'global_amt_mean': float(global_amt_mean), 'global_amt_std': float(global_amt_std)},
            ART_DIR / 'preprocessing_fit_bundle.joblib')

meta = { #small json summary of this run, easy to check what was produced without opening the big parquet files
    'n_rows': int(train.shape[0]),
    'n_features_gbm': int(gbm.shape[1]),
    'n_features_ifae': int(ifae.shape[1]),
    'q60': str(q60), 'q80': str(q80),
    'kaggle_test_file_used': False,
    'split_sizes': train['split'].value_counts().to_dict(),
}
(OUT_DIR / 'preprocessing_meta.json').write_text(json.dumps(meta, indent=2, default=str))

print('Saved:')
print(' -', OUT_DIR / 'train_processed_gbm.parquet', gbm_out.shape)
print(' -', OUT_DIR / 'train_processed_ifae.parquet', ifae_out.shape)
print(' -', ART_DIR / 'preprocessing_fit_bundle.joblib', '(all fitted encoders, for transparency/reuse)')
print(' -', OUT_DIR / 'preprocessing_meta.json')

Saved:
 - /Users/kravchan/Downloads/Supervised Research/project/processed/train_processed_gbm.parquet (590540, 196)
 - /Users/kravchan/Downloads/Supervised Research/project/processed/train_processed_ifae.parquet (590540, 377)
 - /Users/kravchan/Downloads/Supervised Research/project/artifacts/preprocessing_fit_bundle.joblib (all fitted encoders, for transparency/reuse)
 - /Users/kravchan/Downloads/Supervised Research/project/processed/preprocessing_meta.json


## Preprocessing conclusions

- Only labeled data (`train_transaction.csv` + `train_identity.csv`) is used anywhere in this notebook; the unlabeled test files are never loaded.
- The chronological `train`/`val`/`holdout` split is computed first, and every fitted statistic downstream (rare-category vocab, frequency counts, UID aggregates, V-block PCA, one-hot categories, imputation median, scaler mean/std) is fit on `train`-split rows only, then applied to `val`/`holdout` -- no forward leakage across the split boundary.
- The V-block is reduced from 339 raw columns to 11 `Vgrp*_pca1` columns, refit without touching `val` labels.
- Two parallel feature branches are produced: a label-encoded one for the GBM reference model, and a one-hot + frequency-encoded + standardized one for distance/reconstruction-based models such as Isolation Forest and an autoencoder -- so those models are not handed meaningless ordinal integer codes.
- `train_processed_gbm.parquet`, `train_processed_ifae.parquet`, and `preprocessing_meta.json` are the saved outputs of this notebook.

##### End of the notebook